# Advanced Problems with Solutions: Sorting Sequences

Practice advanced Python sorting with `sorted()`, `list.sort()`, `key=`, `reverse=`, stable sorting, dictionary sorting, non-comparable objects, and custom classes.

## Problem 1 — Stable Multi-Criteria Sorting

Given `(name, section, score)` records, sort by:

1. score descending
2. section order: A, B, C
3. original order for exact ties

Solve once with a tuple key and once with stable multi-pass sorting.

In [1]:
records = [
    ("Maya", "B", 91), ("Leo", "A", 88), ("Nora", "A", 91),
    ("Omar", "C", 91), ("Iris", "B", 91), ("Zoe", "A", 88),
    ("Kai", "A", 91),
]
section_rank = {"A": 0, "B": 1, "C": 2}

# Solution A: single tuple key
by_key = sorted(records, key=lambda r: (-r[2], section_rank[r[1]]))

# Solution B: stable multi-pass sorting, least important key first
by_passes = records[:]
by_passes.sort(key=lambda r: section_rank[r[1]])
by_passes.sort(key=lambda r: r[2], reverse=True)

expected = [
    ("Nora", "A", 91), ("Kai", "A", 91),
    ("Maya", "B", 91), ("Iris", "B", 91),
    ("Omar", "C", 91), ("Leo", "A", 88), ("Zoe", "A", 88),
]

assert by_key == expected
assert by_passes == expected
by_key

[('Nora', 'A', 91),
 ('Kai', 'A', 91),
 ('Maya', 'B', 91),
 ('Iris', 'B', 91),
 ('Omar', 'C', 91),
 ('Leo', 'A', 88),
 ('Zoe', 'A', 88)]

### Solution notes

Python's sort is stable, so equal-key records keep their relative order. In the multi-pass version, sort by the secondary key first, then by the primary key.

## Problem 2 — Sort Dictionary Keys by Values

Given product data, return product IDs sorted by:

1. in-stock products first
2. lower price first
3. higher rating first
4. product ID alphabetically

In [2]:
products = {
    "p104": {"in_stock": True,  "price": 18.50, "rating": 4.7},
    "p101": {"in_stock": False, "price": 10.00, "rating": 4.9},
    "p103": {"in_stock": True,  "price": 12.00, "rating": 4.4},
    "p102": {"in_stock": True,  "price": 12.00, "rating": 4.8},
    "p105": {"in_stock": False, "price": 8.00,  "rating": 4.1},
    "p100": {"in_stock": True,  "price": 12.00, "rating": 4.8},
}

sorted_ids = sorted(
    products,
    key=lambda pid: (
        not products[pid]["in_stock"],  # False before True
        products[pid]["price"],
        -products[pid]["rating"],
        pid,
    ),
)

assert sorted_ids == ["p100", "p102", "p103", "p104", "p105", "p101"]
sorted_ids

['p100', 'p102', 'p103', 'p104', 'p105', 'p101']

### Solution notes

Iterating over a dictionary yields keys, so `sorted(products, key=...)` sorts product IDs. Booleans are sortable: `False < True`, which lets `not in_stock` place available products first.

## Problem 3 — Sort Non-Comparable Values

Complex numbers cannot be naturally ordered. Sort the given complex numbers by:

1. distance from origin descending
2. real part ascending
3. imaginary part ascending

In [3]:
points = [3+4j, -3+4j, 1+1j, 0+5j, 2-2j, -1-1j]

sorted_points = sorted(points, key=lambda z: (-abs(z), z.real, z.imag))

assert sorted_points == [(-3+4j), 5j, (3+4j), (2-2j), (-1-1j), (1+1j)]
sorted_points

[(-3+4j), 5j, (3+4j), (2-2j), (-1-1j), (1+1j)]

### Solution notes

The values themselves are not compared directly. The key function maps each complex number to a comparable tuple: `(-abs(z), z.real, z.imag)`.

## Problem 4 — Correctly Choose `sorted()` vs `list.sort()`

Implement `sort_words(words, mutate=False)`.

Rules:

- sort by length ascending
- break ties alphabetically, case-insensitively
- when `mutate=False`, return a new list and do not change the input
- when `mutate=True`, mutate the original list and return `None`

In [4]:
def sort_words(words, mutate=False):
    key = lambda word: (len(word), word.casefold())
    if mutate:
        words.sort(key=key)
        return None
    return sorted(words, key=key)


words = ["Banana", "fig", "apple", "Date", "kiwi", "pear", "apricot"]
original_id = id(words)

new_result = sort_words(words)
assert new_result == ["fig", "Date", "kiwi", "pear", "apple", "Banana", "apricot"]
assert words == ["Banana", "fig", "apple", "Date", "kiwi", "pear", "apricot"]
assert id(words) == original_id

mutating_result = sort_words(words, mutate=True)
assert mutating_result is None
assert words == ["fig", "Date", "kiwi", "pear", "apple", "Banana", "apricot"]
assert id(words) == original_id

words

['fig', 'Date', 'kiwi', 'pear', 'apple', 'Banana', 'apricot']

### Solution notes

Use `sorted()` when you need a new list. Use `list.sort()` when mutation is intended. `list.sort()` returns `None` by design.

## Problem 5 — Custom Class Natural Ordering

Create a `Task` class. `sorted(tasks)` should order tasks by:

1. priority ascending; smaller means more urgent
2. duration ascending
3. title alphabetically, case-insensitively

Then show the equivalent `key=` solution.

In [5]:
from functools import total_ordering

@total_ordering
class Task:
    def __init__(self, title, priority, minutes):
        self.title = title
        self.priority = priority
        self.minutes = minutes

    def sort_key(self):
        return (self.priority, self.minutes, self.title.casefold())

    def __lt__(self, other):
        if not isinstance(other, Task):
            return NotImplemented
        return self.sort_key() < other.sort_key()

    def __eq__(self, other):
        if not isinstance(other, Task):
            return NotImplemented
        return self.sort_key() == other.sort_key()

    def __repr__(self):
        return f"Task({self.title!r}, priority={self.priority}, minutes={self.minutes})"


tasks = [
    Task("Write report", 2, 60),
    Task("Fix outage", 1, 45),
    Task("Email team", 2, 10),
    Task("Backup data", 1, 30),
    Task("book flights", 2, 10),
]

natural = sorted(tasks)
key_based = sorted(tasks, key=lambda t: (t.priority, t.minutes, t.title.casefold()))

expected_titles = ["Backup data", "Fix outage", "book flights", "Email team", "Write report"]
assert [t.title for t in natural] == expected_titles
assert [t.title for t in key_based] == expected_titles
natural

[Task('Backup data', priority=1, minutes=30),
 Task('Fix outage', priority=1, minutes=45),
 Task('book flights', priority=2, minutes=10),
 Task('Email team', priority=2, minutes=10),
 Task('Write report', priority=2, minutes=60)]

### Solution notes

Implementing `__lt__` gives the class a natural ordering. A `key=` function is often preferable when there are several possible valid orderings for the same class.

## Best-Practice Checklist

- Prefer `key=` over comparison-style sorting.
- Use tuple keys for multi-criteria sorting.
- Use negative numeric keys for descending sub-sorts.
- Rely on stable sorting to preserve tie order.
- Use `sorted()` for a new list and `list.sort()` for in-place mutation.
- Give custom classes natural ordering only when one default ordering is truly obvious.